# Lesson 11: Interventions in an experiment

**You will learn:** how to attach interventions to an `Experiment`, so each condition runs on a changed subject while the same recorders measure it.
**Requires:** CPU only, about a minute. Read [Lesson 10](10_run_an_experiment.ipynb) first.

Lesson 10 ran one condition. Here the conditions differ in what is done to the subject: `Intervene` applies a `StateChange` through the subject's own `intervention` at the start of each matching repetition and undoes it at the end. Each stimulus is presented once per condition, so the recording and the answer come from that one presentation.

**The question:** this is Lesson 8's lesion study run as one experiment, with three conditions (intact, animal-selective units silenced, random units silenced), recording IT as well as the answers.

## Setup

Imports and settings that are not part of the method. The cell is folded; click it to expand.

In [1]:
# Keep the output short: no progress bars, no library warnings.
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
import tqdm as _tqdm
for _bar in (tqdm, _tqdm.tqdm):
    _bar.__init__ = partialmethod(_bar.__init__, disable=True)
warnings.filterwarnings('ignore')
import os
os.environ['BS_INSTALL_DEPENDENCIES'] = 'no'      # every requirement is already installed

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image

ASSETS = Path('../assets/lessons')        # weights and images committed with the lessons

# SqueezeNet 1.1 trained on ImageNet (5 MB). These images come from another dataset, CIFAR-10.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()                                # inference only: no dropout, no training

# The network expects 224 x 224 images, normalized the way it was trained.
TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files -> one (batch, 3, 224, 224) array."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

## The subject

A `BrainScoreModel` is assembled from parts. The **extractor** prepares images and records layers; the **region map** says which layer stands for which brain area (a hypothesis, not a fact). `behavioral_readout_layer` names the layer a readout answers from ([Lesson 6](06_behavior.ipynb)). `intervention` lets an experiment change the network during a run ([Lesson 7](07_perturb_a_model.ipynb)).

In [2]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'},
        behavioral_readout_layer='features.12',       # answers are read out from this layer
        intervention=build_pytorch_ablation_fn(network))   # carries out requested changes

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(subject.out_channels)                          # what this subject can produce

['behavior',
 'behavior:choice',
 'behavior:probabilities',
 'behavior:rating',
 'behavior:rt',
 'neural:IT',
 'neural:V1',
 'neural:V4',
 'perturbation']

## The experiment

### Stimuli

In [3]:
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images, 12 of each CIFAR-10 category, each labelled animal or vehicle.
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [4]:
# Half the images (6 per category) fit the readout; the other half test it.
position = stimuli.groupby('category').cumcount()          # 0..11 within each category
train = StimulusSet(stimuli[position < 6].reset_index(drop=True)); train.identifier = 'lessons-train'
test = StimulusSet(stimuli[position >= 6].reset_index(drop=True)); test.identifier = 'lessons-test'
for part in (train, test):
    part.stimulus_paths = dict(zip(part['stimulus_id'], part['image_path']))
len(train), len(test)

(60, 60)

### Task

The subject answers "animal" or "vehicle" and also gives the probability of each. The readout is fitted once, on the intact network, and kept for every condition.

In [5]:
from brainscore_core import TaskContext
from brainscore_core.responses import Choice

task = TaskContext(response=Choice(['animal', 'vehicle'], probabilities=True), fitting_stimuli=train,
                   prefer_path='readout', readout_fit='intact')

### Conditions

The units are chosen as in Lesson 8: 2,000 animal-selective units of `features.12` found on the training images, and 2,000 random units from the same population. Each silencing becomes an `Intervene` tool attached to one condition.

In [6]:
from brainscore_core import Perturbation, StateChange
from brainscore_core.selection import FunctionalSelection, RandomSelection
from brainscore.experiments import Intervene

K = 2000
animal_units = FunctionalSelection('IT', train, contrast=(['animal'], ['vehicle']),
                                   contrast_column='label', n_units=K).resolve(subject)

# Draw the random units from the same population, in the same layout.
population = animal_units.metadata['unit_population']
random_units = RandomSelection(animal_units.layer, n_units=K, n_total=len(population),
                               population=population, seed=0, axis=animal_units.axis,
                               expected_shape=animal_units.expected_shape).resolve(subject)
subject.reset()                                 # the localizer run leaves nothing behind

def silence(selection):
    """A request to set these units to zero."""
    return StateChange(kind='ablation', target=selection, perturbation=Perturbation(kind='zero'))

CONDITIONS = ['intact', 'animal_units', 'random_units']
interventions = [
    Intervene(silence(animal_units), conditions=['animal_units'], name='silence_animal_units'),
    Intervene(silence(random_units), conditions=['random_units'], name='silence_random_units')]

## Run

The protocol presents the test images once per condition. The recorders come first in the tool list, then the interventions.

In [7]:
import tempfile
from brainscore.experiments import Experiment, RecordBehavior, RecordNeural, TaskProtocol

experiment = Experiment(
    subject=subject,
    protocol=TaskProtocol(test, task, conditions=CONDITIONS),
    tools=[RecordNeural(['IT']), RecordBehavior()] + interventions,
    output_dir=tempfile.mkdtemp() + '/run')     # a new, empty folder for this run
experiment.validate()     # checks subject, channels and tools without running the model
result = experiment.run()
result.manifest['status']

'complete'

## Read results back

The accessors read the saved assemblies; they do not run the model again.

In [8]:
choices = {condition: result.behavior(condition=condition) for condition in CONDITIONS}
probabilities = {condition: result.behavior(condition=condition, measure='probabilities')
                 for condition in CONDITIONS}
it = {condition: result.neural('IT', condition=condition) for condition in CONDITIONS}
it['animal_units'].attrs['recording']

{'task_type': 'choice',
 'path': 'readout',
 'response': 'Choice',
 'forward_pass': 'readout extraction',
 'readout_fit': 'intact',
 'condition': 'animal_units'}

Each recording says how it was made. The readout was fitted on the intact network and tested on each condition; pass `TaskProtocol(..., refit_readout=True)` to refit it on the damaged network instead, which asks what information survives rather than what the trained readout can still do.

## Analysis

Plain Python from here on: no Brain-Score calls.

In [9]:
LABELS = np.array(['animal', 'vehicle'])
truth = np.asarray(test['label'])

def accuracy(choice_values):
    """Fraction of test images whose chosen label is the true one."""
    chosen = LABELS[np.asarray(choice_values).argmax(axis=1)]
    return float(np.mean(chosen == truth))

def mean_p_correct(probability_values):
    """Average probability the subject gave to the true label."""
    true_column = np.where(truth == 'animal', 0, 1)            # column 0 = animal, 1 = vehicle
    return float(np.asarray(probability_values)[np.arange(len(truth)), true_column].mean())

def mean_magnitude(values):
    """Average absolute activity across images and units."""
    return float(np.abs(np.asarray(values)).mean())

## Results

In [10]:
rows = []
for condition in CONDITIONS:
    rows.append({'condition': condition,
                 'accuracy': accuracy(choices[condition].values),
                 'mean p(correct)': mean_p_correct(probabilities[condition].values),
                 'mean |IT|': mean_magnitude(it[condition].values)})
pd.DataFrame(rows).set_index('condition').round(3)

,accuracy,mean p(correct),mean |IT|
condition,,,
intact,0.833,0.792,1.725
animal_units,0.833,0.779,1.645
random_units,0.833,0.789,1.683


Accuracy is 0.833 in all three conditions; the animal-selective lesion lowers the probability given to the correct answer slightly more (0.792 to 0.779) than the random lesion (0.789), and lowers IT activity more.

## Summary

- The task says what the subject does (`'passive'` is plain viewing); the recorders say what is measured, and one presentation gives both.
- `Intervene` applies a change through the subject's own `intervention` per condition and undoes it afterwards.
- `result.neural(region, condition=...)` and `result.behavior(condition=..., measure=...)` read back assemblies.

You have now seen every core concept. The [experiments](../experiments/) put them to use on scientific questions.